# Integration language

Anton Antonov   
October 2026

----

## Setup

In [10]:
use FunctionalParsers :ALL;
use FunctionalParsers::EBNF;

---

## Commands

In [107]:
my @commands =
   "compute the symbolic integral of Sin[x]*x^2 over 0 to 10",
   "integrate symbolically Sin[x]*x^2 over a b",
   "what is the integral of t^4+Log[t] for t in the interval 0 to a",
   "compute the numerical integral of Sin[x] for x over 0 to 10",
   "integrate numerically Sin[x^2]+5 in 0 to 10",
   "integrate numerically 1/x+Sqrt[x] in the interval 4 and 10",
   "integrate x^2+5",
   "integrate Exp[-x] over R+",
   "compute the integral of Log[x] of x from 0 to 10",
   "compute the integral of Log[x] of x from 0 to 10 and plot it",
   "integrate and plot Sin[x]*x^2 over 0 1"
;

deduce-type(@commands)

Vector(Atom((Str)), 11)

---

## Grammar (functional parsers)

In [ ]:
sink my $integrationGrammar = q:to/END/;
<request-preamble> = 'compute' | 'what' , 'is' ;
<compute> = <request-preamble> , [ 'the' ] ;
<integral-type> = [ 'numerical' | 'symbolic' ] <& 'integral' <@ IType ;
<integrate> = [ 'symbolically' | 'numerically' ] <& 'integrate' | 'integrate' &> ( 'numerically' | 'symbolically' ) <@ IType ;
<function> = '_String' <@ IFunc ;
<var> = '_IdentifierString' <@ IVar ;
<range-named> =  'R' | 'R+' | 'R-' <@ IRange ;
<infinity> = { '+' | '-' } , ( 'Inf' | 'Infinity' | 'inf' | '∞' ) ;
<range-boundary> = '_Number' | '_IdentifierString' | <infinity> ;
<range-interval> = { 'from' } &> <range-boundary> , [ 'to' | 'and' ] &> <range-boundary> <@ IRange ;
<range> = ( [ 'in' | 'over' ] , [ 'the' ] , [ 'interval' ] ) &> ( <range-interval> | <range-named> ) ;
<var-range-spec> = ( 'for' | 'of' ) &> <var> , <range> ;
<integral-request> = <compute> &> <integral-type> <& 'of' ;
<command1> = ( <integral-request> | <integrate> ) , <function> , ( <var-range-spec> | <range> ) ;
<command2> = ( <integral-request> | <integrate> ) , <function> ;
<command> = <command1> | <command2> <@ ICommand ;
END

In [123]:
sink my $integrationGrammar = q:to/END/;
<request-preamble> = 'compute' | 'what' , 'is' ;
<compute> = <request-preamble> , [ 'the' ] ;
<integral-type> = [ 'numerical' | 'symbolic' ] <& 'integral' <@ IType ;
<integrate> = [ 'symbolically' | 'numerically' ] <& 'integrate' | 'integrate' &> ( 'numerically' | 'symbolically' ) <@ IType ;
<function> = '_String' <@ IFunc ;
<var> = '_IdentifierString' <@ IVar ;
<range-named> =  'R' | 'R+' | 'R-' <@ IRange ;
<infinity> = { '+' | '-' } , ( 'Inf' | 'Infinity' | 'inf' | '∞' ) ;
<range-boundary> = '_Number' | '_IdentifierString' | <infinity> ;
<range-interval> = { 'from' } &> <range-boundary> , [ 'to' | 'and' ] &> <range-boundary> <@ IRange ;
<range> = ( [ 'in' | 'over' ] , [ 'the' ] , [ 'interval' ] ) &> ( <range-interval> | <range-named> ) ;
<var-range-spec> = ( 'for' | 'of' ) &> <var> , <range> ;
<integral-request> = <compute> &> <integral-type> <& 'of' ;
<plot> = 'plot' <& [ 'it' ] <@ IPlot ;
<command1> = ( <integral-request> | <integrate> , [ 'and' &> <plot> ] ) , <function> , ( <var-range-spec> | <range> ) , [ 'and' &> <plot> ] ;
<command2> = ( <integral-request> | <integrate> ) , <function> ;
<command> = <command1> | <command2> <@ ICommand ;
END

In [102]:
fp-ebnf-parse($integrationGrammar).head.tail

class FP {
	method pREQUEST-PREAMBLE(@x) { alternatives(symbol('compute'), sequence(symbol('what'), symbol('is')))(@x) }
	method pCOMPUTE(@x) { sequence({self.pREQUEST-PREAMBLE($_)}, option(symbol('the')))(@x) }
	method pINTEGRAL-TYPE(@x) { apply(&IType, sequence-pick-left(option(alternatives(symbol('numerical'), symbol('symbolic'))), symbol('integral')))(@x) }
	method pINTEGRATE(@x) { alternatives(sequence-pick-left(option(alternatives(symbol('symbolically'), symbol('numerically'))), symbol('integrate')), apply(&IType, sequence-pick-right(symbol('integrate'), (alternatives(symbol('numerically'), symbol('symbolically'))))))(@x) }
	method pFUNCTION(@x) { apply(&IFunc, symbol('_String'))(@x) }
	method pVAR(@x) { apply(&IVar, symbol('_IdentifierString'))(@x) }
	method pRANGE-NAMED(@x) { alternatives(symbol('R'), symbol('R+'), apply(&IRange, symbol('R-')))(@x) }
	method pINFINITY(@x) { sequence(many(alternatives(symbol('+'), symbol('-'))), option(alternatives(symbol('Inf'), symbol('Infinit

----

## Generation (Grammar)


In [124]:
my $code = fp-ebnf-parse($integrationGrammar, <CODE>, actions => 'Raku::Grammar', name => 'Integration', prefix => '').head.tail;

grammar Integration {
	rule REQUEST-PREAMBLE { 'compute' | 'what' 'is' }
	rule COMPUTE { <REQUEST-PREAMBLE> 'the'? }
	rule INTEGRAL-TYPE { ['numerical' | 'symbolic']? 'integral' }
	rule INTEGRATE { ['symbolically' | 'numerically']? 'integrate' | 'integrate' ['numerically' | 'symbolically'] }
	rule FUNCTION { '_String' }
	rule VAR { '_IdentifierString' }
	rule RANGE-NAMED { 'R' | 'R+' | 'R-' }
	rule INFINITY { ['+' | '-']* ['Inf' | 'Infinity' | 'inf' | '∞'] }
	rule RANGE-BOUNDARY { '_Number' | '_IdentifierString' | <INFINITY> }
	rule RANGE-INTERVAL { 'from'* <RANGE-BOUNDARY> ['to' | 'and']? <RANGE-BOUNDARY> }
	rule RANGE { [['in' | 'over']? 'the'? 'interval'?] [<RANGE-INTERVAL> | <RANGE-NAMED>] }
	rule VAR-RANGE-SPEC { ['for' | 'of'] <VAR> <RANGE> }
	rule INTEGRAL-REQUEST { <.COMPUTE> <INTEGRAL-TYPE> 'of' }
	rule PLOT { 'plot' 'it'? }
	rule COMMAND1 { [<INTEGRAL-REQUEST> | <INTEGRATE> ['and' <PLOT>]?] <FUNCTION> [<VAR-RANGE-SPEC> | <RANGE>] ['and' <PLOT>]? }
	rule COMMAND2 { [<INTEGRAL-

In [166]:
my grammar GIntegration {
	rule REQUEST-PREAMBLE { 'compute' | 'what' 'is' }
	rule COMPUTE { <REQUEST-PREAMBLE> 'the'? }
	rule INTEGRAL-TYPE { ['numerical' | 'symbolic']? 'integral' }
	rule INTEGRATE { ['symbolically' | 'numerically']? 'integrate' | 'integrate' ['numerically' | 'symbolically'] }
	rule FUNCTION { \S+ }
	regex VAR { <.alpha> <.alnum>* }
	rule RANGE-NAMED { 'R' | 'R+' | 'R-' }
	regex INFINITY {:i ['+' | '-']? ['Inf' | 'Infinity' | 'inf' | '∞'] }
	regex number { <[+\-]>? \d+ [ <[.Ee]> <[+\-]>? \d+]? }
	regex RANGE-BOUNDARY { <number> | <VAR> | <INFINITY> }
	regex RANGE-INTERVAL { ['from' \s+]? <RANGE-BOUNDARY> [\s+ ['to' | 'and']]? \s+ <RANGE-BOUNDARY> }
	regex RANGE { [['in' | 'over']? [\s+ 'the']? [\s+ 'interval']? \s+]? [<RANGE-INTERVAL> | <RANGE-NAMED>] }
	rule VAR-RANGE-SPEC { ['for' | 'of'] <VAR> <RANGE> }
	rule INTEGRAL-REQUEST { <.COMPUTE> <INTEGRAL-TYPE> 'of' }
	rule PLOT { 'plot' 'it'? }
	rule COMMAND1 { [<INTEGRAL-REQUEST> | <INTEGRATE> ['and' <PLOT>]?] <FUNCTION> [<VAR-RANGE-SPEC> | <RANGE>] ['and' <PLOT>]? }
	rule COMMAND2 { [<INTEGRAL-REQUEST> | <INTEGRATE>] <FUNCTION> }
	rule COMMAND { <COMMAND1> | <COMMAND2> }
	rule TOP { <COMMAND> }
}

(GIntegration)

### Parsing experiments

In [156]:
@commands[9]

compute the integral of Log[x] of x from 0 to 10 and plot it

In [169]:
my $p = GIntegration.new;
GIntegration.subparse(@commands[9])

｢compute the integral of Log[x] of x from 0 to 10 and plot it｣
 COMMAND => ｢compute the integral of Log[x] of x from 0 to 10 and plot it｣
  COMMAND1 => ｢compute the integral of Log[x] of x from 0 to 10 and plot it｣
   INTEGRAL-REQUEST => ｢compute the integral of ｣
    INTEGRAL-TYPE => ｢integral ｣
   FUNCTION => ｢Log[x] ｣
   VAR-RANGE-SPEC => ｢of x from 0 to 10 ｣
    VAR => ｢x｣
    RANGE => ｢from 0 to 10｣
     RANGE-INTERVAL => ｢from 0 to 10｣
      RANGE-BOUNDARY => ｢0｣
       number => ｢0｣
      RANGE-BOUNDARY => ｢10｣
       number => ｢10｣
   PLOT => ｢plot it｣

In [168]:
#GIntegration.subparse('from 0 to 10', rule => 'RANGE')
GIntegration.subparse('for x from 0 to 10', rule => 'VAR-RANGE-SPEC')

｢for x from 0 to 10｣
 VAR => ｢x｣
 RANGE => ｢from 0 to 10｣
  RANGE-INTERVAL => ｢from 0 to 10｣
   RANGE-BOUNDARY => ｢0｣
    number => ｢0｣
   RANGE-BOUNDARY => ｢10｣
    number => ｢10｣

---

## Generation (FP)

Preview of the code:

In [83]:
my $code = fp-ebnf-parse($integrationGrammar, <CODE>, actions => 'Raku::Class', name => 'Integration', prefix => 'p').head.tail;

class Integration {
	method pREQUEST-PREAMBLE(@x) { alternatives(symbol('compute'), sequence(symbol('what'), symbol('is')))(@x) }
	method pCOMPUTE(@x) { sequence({self.pREQUEST-PREAMBLE($_)}, option(symbol('the')))(@x) }
	method pINTEGRAL-TYPE(@x) { apply(&IType, sequence-pick-left(option(alternatives(symbol('numerical'), symbol('symbolic'))), symbol('integral')))(@x) }
	method pINTEGRATE(@x) { alternatives(sequence-pick-left(option(alternatives(symbol('symbolically'), symbol('numerically'))), symbol('integrate')), apply(&IType, sequence-pick-right(symbol('integrate'), (alternatives(symbol('numerically'), symbol('symbolically'))))))(@x) }
	method pFUNCTION(@x) { apply(&IFunc, symbol('_String'))(@x) }
	method pVAR(@x) { apply(&IVar, symbol('_IdentifierString'))(@x) }
	method pRANGE-NAMED(@x) { alternatives(symbol('R'), symbol('R+'), apply(&IRange, symbol('R-')))(@x) }
	method pRANGE-INTERVAL(@x) { apply(&IRange, sequence-pick-right(many(symbol('from')), sequence(symbol('_WordString'), s

In [84]:
$code
==> {
    .subst("symbol('_WordString')", 'self.pWord', :g)
    .subst("symbol('_Number')", 'self.pNumber', :g)
    .subst("symbol('_String')", 'self.pString', :g)
    .subst("symbol('_IdentifierString')", 'self.pIdentifier', :g)
    .subst(/ '&' (pI \w+) /, { 'self.' ~ $0 }, :g)
    .subst(/ '&' (I \w+) /, { 'self.' ~ $0 }, :g)
}()

class Integration {
	method pREQUEST-PREAMBLE(@x) { alternatives(symbol('compute'), sequence(symbol('what'), symbol('is')))(@x) }
	method pCOMPUTE(@x) { sequence({self.pREQUEST-PREAMBLE($_)}, option(symbol('the')))(@x) }
	method pINTEGRAL-TYPE(@x) { apply(self.IType, sequence-pick-left(option(alternatives(symbol('numerical'), symbol('symbolic'))), symbol('integral')))(@x) }
	method pINTEGRATE(@x) { alternatives(sequence-pick-left(option(alternatives(symbol('symbolically'), symbol('numerically'))), symbol('integrate')), apply(self.IType, sequence-pick-right(symbol('integrate'), (alternatives(symbol('numerically'), symbol('symbolically'))))))(@x) }
	method pFUNCTION(@x) { apply(self.IFunc, self.pString)(@x) }
	method pVAR(@x) { apply(self.IVar, self.pIdentifier)(@x) }
	method pRANGE-NAMED(@x) { alternatives(symbol('R'), symbol('R+'), apply(self.IRange, symbol('R-')))(@x) }
	method pRANGE-INTERVAL(@x) { apply(self.IRange, sequence-pick-right(many(symbol('from')), sequence(self.pWord, sequ

In [94]:
my role Context {
    has &.ICommand = sub ($x) { ['ICommand', $x] }
    has &.IFunc = sub ($x) { ['IFunc', $x] }
    has &.IRange = sub ($x) { ['IRange', $x] }
    has &.IType = sub ($x) { ['IType', $x] }
    has &.IVar = sub ($x) { ['IVar', $x] }
    has &.IPlot = sub ($x) { ['IPlot', $x] }

    has &.pWord = apply({ $_.flat.join }, satisfy({ $_.join(' ') ~~ / ^ \w+ $ /}));
    has &.pString = apply({ $_.flat.join }, satisfy({ $_.join(' ') ~~ / ^ \S+ $ /}));
    has &.pNumber = apply({ $_.join.Num }, satisfy({ $_.join(' ') ~~ / ^ [ \d | '.' | 'e' | 'E' ]+ $ /}));
    has &.pIdentifier = apply({ $_.flat.join }, satisfy({$_.join(' ') ~~ / ^ <alpha> <alnum>* $ /}));
}

(Context)

In [95]:
my class Integration does Context {
	method pREQUEST-PREAMBLE(@x) { alternatives(symbol('compute'), sequence(symbol('what'), symbol('is')))(@x) }
	method pCOMPUTE(@x) { sequence({self.pREQUEST-PREAMBLE($_)}, option(symbol('the')))(@x) }
	method pINTEGRAL-TYPE(@x) { apply(self.IType, sequence-pick-left(option(alternatives(symbol('numerical'), symbol('symbolic'))), symbol('integral')))(@x) }
	method pINTEGRATE(@x) { alternatives(sequence-pick-left(option(alternatives(symbol('symbolically'), symbol('numerically'))), symbol('integrate')), apply(self.IType, sequence-pick-right(symbol('integrate'), (alternatives(symbol('numerically'), symbol('symbolically'))))))(@x) }
	method pFUNCTION(@x) { apply(self.IFunc, self.pString)(@x) }
	method pVAR(@x) { apply(self.IVar, self.pIdentifier)(@x) }
	method pRANGE-NAMED(@x) { alternatives(symbol('R'), symbol('R+'), apply(self.IRange, symbol('R-')))(@x) }
	method pRANGE-INTERVAL(@x) { apply(self.IRange, sequence-pick-right(many(symbol('from')), sequence(self.pWord, sequence-pick-right(option(alternatives(symbol('to'), symbol('and'))), self.pWord))))(@x) }
	method pRANGE(@x) { sequence-pick-right((sequence(option(alternatives(symbol('in'), symbol('over'))), sequence(option(symbol('the')), option(symbol('interval'))))), (alternatives({self.pRANGE-INTERVAL($_)}, {self.pRANGE-NAMED($_)})))(@x) }
	method pVAR-RANGE-SPEC(@x) { sequence-pick-right((alternatives(symbol('for'), symbol('of'))), sequence({self.pVAR($_)}, {self.pRANGE($_)}))(@x) }
	method pINTEGRAL-REQUEST(@x) { sequence-pick-right({self.pCOMPUTE($_)}, sequence-pick-left({self.pINTEGRAL-TYPE($_)}, symbol('of')))(@x) }
	method pPLOT(@x) { apply(self.IPlot, sequence-pick-left(symbol('plot'), option(symbol('it'))))(@x) }
	method pCOMMAND1(@x) { sequence((alternatives({self.pINTEGRAL-REQUEST($_)}, sequence({self.pINTEGRATE($_)}, option(sequence-pick-right(symbol('and'), {self.pPLOT($_)}))))), sequence({self.pFUNCTION($_)}, sequence((alternatives({self.pVAR-RANGE-SPEC($_)}, {self.pRANGE($_)})), option(sequence-pick-right(symbol('and'), {self.pPLOT($_)})))))(@x) }
	method pCOMMAND2(@x) { sequence((alternatives({self.pINTEGRAL-REQUEST($_)}, {self.pINTEGRATE($_)})), {self.pFUNCTION($_)})(@x) }
	method pCOMMAND(@x) { shortest(alternatives({self.pCOMMAND1($_)}, apply(self.ICommand, {self.pCOMMAND2($_)})))(@x) }
	method FALLBACK ($name, |c) { "$name\(\)" }
	has &.parser is rw = -> @x { self.pTOP(@x) };
}

In [97]:
my $p = Integration.new;
.say for |$p.pCOMMAND(@commands.head.words)

(() ([IType (symbolic)] ([IFunc Sin[x]*x^2] ([IRange (0 10)] ()))))


In [98]:
#% html
my @field-names = <command parsed residual>;
my @tbl = @commands.map({ my @res = |$p.pCOMMAND($_.words); @field-names Z=> ($_, @res.head.tail.gist, @res.head.head.gist) })».Hash;
@tbl
==> to-html(:@field-names)

command,parsed,residual
compute the symbolic integral of Sin[x]*x^2 over 0 to 10,([IType (symbolic)] ([IFunc Sin[x]*x^2] ([IRange (0 10)] ()))),()
integrate symbolically Sin[x]*x^2 over a b,(([IType symbolically] ()) ([IFunc Sin[x]*x^2] ([IRange (a b)] ()))),()
what is the integral of t^4+Log[t] for t in the interval 0 to a,([IType ()] ([IFunc t^4+Log[t]] (([IVar t] [IRange (0 a)]) ()))),()
compute the numerical integral of Sin[x] for x over 0 to 10,([IType (numerical)] ([IFunc Sin[x]] (([IVar x] [IRange (0 10)]) ()))),()
integrate numerically Sin[x^2]+5 in 0 to 10,(([IType numerically] ()) ([IFunc Sin[x^2]+5] ([IRange (0 10)] ()))),()
integrate numerically 1/x+Sqrt[x] in the interval 4 and 10,(([IType numerically] ()) ([IFunc 1/x+Sqrt[x]] ([IRange (4 10)] ()))),()
integrate x^2+5,[ICommand (() [IFunc x^2+5])],()
integrate Exp[-x] over R+,((() ()) ([IFunc Exp[-x]] (R+ ()))),()
compute the integral of Log[x] of x from 0 to 10,([IType ()] ([IFunc Log[x]] (([IVar x] [IRange (0 10)]) ()))),()
compute the integral of Log[x] of x from 0 to 10 and plot it,([IType ()] ([IFunc Log[x]] (([IVar x] [IRange (0 10)]) ([IPlot plot])))),()
